# 03 — Working with BEAST Output

This notebook demonstrates how to style trees generated by [BEAST](https://beast.community/) for publication.

BEAST outputs typically include:
- Posterior support values on internal nodes
- Branch rates as annotations
- Height/age annotations

FigTreeKit can configure FigTree to display these annotations.

In [ ]:
from figtreekit import FigTreeStyler, LayoutType
from pathlib import Path

Path("output").mkdir(exist_ok=True)

## 1. Simulate a BEAST-like Tree

In practice, you would load a BEAST `.trees` file. Here we simulate one with annotations.

In [ ]:
# Create a tree with BEAST-style annotations in the Nexus comment format
beast_nexus = """#NEXUS
Begin taxa;
    Dimensions ntax=6;
    Taxlabels
        Seq_A
        Seq_B
        Seq_C
        Seq_D
        Seq_E
        Seq_F
    ;
End;

Begin trees;
    Tree tree1 = [&R] ((Seq_A:0.1[&posterior=1.0,rate=0.002],Seq_B:0.15[&posterior=0.95,rate=0.003]):0.2[&posterior=0.85],(Seq_C:0.05[&posterior=1.0,rate=0.001],(Seq_D:0.08[&posterior=0.9,rate=0.004],Seq_E:0.12[&posterior=0.88,rate=0.002]):0.1[&posterior=0.75]):0.25[&posterior=0.92],Seq_F:0.3[&posterior=1.0,rate=0.001]):0.05;
End;
"""

Path("output/beast_example.nex").write_text(beast_nexus)
print("Created simulated BEAST Nexus file.")

## 2. Load and Configure for BEAST Output

In [ ]:
styler = FigTreeStyler("output/beast_example.nex")
print(f"Loaded BEAST tree (Nexus format: {styler._is_nexus_format})")

## 3. Display Posterior Support Values

Configure node labels to show the `posterior` attribute from BEAST annotations.

In [ ]:
styler.set_node_labels(
    is_shown=True,
    display_attribute="posterior",
    font_size=8,
    font_name="Arial"
)
print("Node labels configured to show posterior support.")

## 4. Color Branches by Substitution Rate

Use `branch_color_attribute` to create a color gradient based on the `rate` annotation.

In [ ]:
styler.set_appearance(
    branch_line_width=2.0,
    branch_color_attribute="rate"  # Color branches by substitution rate
)
print("Branches will be colored by substitution rate.")

## 5. Configure Polar Layout for Time-Calibrated Tree

In [ ]:
styler.set_layout(LayoutType.POLAR)
styler.set_polar_layout(
    angular_range=360,
    root_angle=90,
    align_tip_labels=True
)

# Show scale axis for time calibration
styler.set_scale_axis(is_shown=True)
styler.set_scale_bar(is_shown=True)

## 6. Style Tip Labels

In [ ]:
styler.set_tip_labels(
    is_shown=True,
    font_name="Helvetica",
    font_size=10
)

## 7. Export for FigTree

In [ ]:
output_path = "output/03_beast_styled.nex"
styler.export(output_path)
print(f"Exported to: {output_path}")
print(f"File size: {Path(output_path).stat().st_size} bytes")
print("\nOpen this file in FigTree to see the styled tree.")

## 8. Validate Before Export

Use `validate()` to check for potential issues before committing to an export.

In [ ]:
issues = styler.validate()
if issues:
    print("Validation issues found:")
    for issue in issues:
        print(f"  - {issue}")
else:
    print("No validation issues. Tree is FigTree-compatible.")

## Summary

For BEAST output styling:

1. Load the `.trees` or `.nex` file
2. Set `node_labels.display_attribute = "posterior"` for support values
3. Set `appearance.branch_color_attribute = "rate"` for rate coloring
4. Choose layout (polar works well for time-calibrated trees)
5. Export and open in FigTree

All BEAST annotations (`[&key=value]`) are preserved through the round-trip.